# LayerCAM vs FPN-LayerCAM vs GradCAM vs FPN-GradCAM
## 2×2 Factorial Comparison on PanNuke — DenseNet169 Multi-Label Classification

This notebook implements a clean **2×2 factorial design** that isolates two independent design axes:

| | **Single Layer** (`denseblock4`) | **Feature Pyramid** (db1 + db2 + db4) |
|---|---|---|
| **GAP gradient** (GradCAM) | Standard GradCAM | FPN-GradCAM |
| **Pixel-wise gradient** (LayerCAM) | Standard LayerCAM | FPN-LayerCAM |

**LayerCAM** [Jiang et al., 2021] retains full spatial gradient information before fusion:
$$L^c_{\text{LayerCAM}} = \text{ReLU}\!\left(\sum_k \underbrace{\text{ReLU}\!\left(\frac{\partial Y^c}{\partial A_k}\right)}_{\text{no GAP — full } h\!\times\!w} \!\odot\, A_k\right)$$

vs **GradCAM** which spatially averages the gradient first:
$$L^c_{\text{GradCAM}} = \text{ReLU}\!\left(\sum_k \underbrace{\frac{1}{Z}\sum_{ij}\frac{\partial Y^c}{\partial A_k^{ij}}}_{\alpha_k^c\ =\ \text{scalar}} \cdot A_k\right)$$

The factorial design cleanly separates the contributions of each axis.
If FPN-LayerCAM wins, *both* axes matter and their effects are additive.
If Standard LayerCAM beats FPN-GradCAM, gradient treatment matters more than resolution.

**Run order prerequisite:** `pannuke_densenet169.ipynb` must have produced:
- `outputs/best_densenet169_pannuke.pth`
- `outputs/test_predictions.json`
- `outputs/test_images/`

---
### Outputs (`outputs/layercam_comparison/`)

| File | Description |
|---|---|
| `triplets/{method}/` | Per-image triplet figures — original / GT mask / CAM overlay |
| `results_50_test.csv` | IoU @ 3 thresholds × 4 methods × 50 test images |
| `method_iou_summary.csv` | Macro IoU table (main paper table) |
| `results_full_dataset.csv` | Per-image IoU × 4 methods — full dataset (correctly classified) |
| `per_class_n1_vs_nmulti.csv` | n=1 vs n≥2 per class per method with Welch's t-test |
| `pairwise_interference_all_methods.csv` | ΔIoU(c∣c') × 4 methods |
| `permutation_results_all_methods.csv` | Empirical p-values (N=1000) |
| `robustness_layercam_vs_gradcam.csv` | LayerCAM robustness gain per pair, Wilcoxon test |
| `figures/fig{1..10}.*` | 10 publication-ready figures (PDF + PNG @ 300 DPI) |

## 1. Imports & Environment

In [ ]:
import gc
import json
import random
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import cv2
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Patch
import seaborn as sns
from tqdm.auto import tqdm
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
import torchvision.models as models
from torch.utils.data import Dataset

warnings.filterwarnings('ignore')
matplotlib.rcParams.update({
    'figure.dpi'   : 120,
    'font.family'  : 'DejaVu Sans',
    'axes.spines.top'   : False,
    'axes.spines.right' : False,
})

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Configuration

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────────
BASE_DIR     = Path('.')
OUTPUT_ROOT  = Path('./outputs')
MODEL_PATH   = OUTPUT_ROOT / 'best_densenet169_pannuke.pth'
TEST_JSON    = OUTPUT_ROOT / 'test_predictions.json'
TEST_IMG_DIR = OUTPUT_ROOT / 'test_images'

OUT_DIR  = OUTPUT_ROOT / 'layercam_comparison'
FIG_DIR  = OUT_DIR / 'figures'
TRIP_DIR = OUT_DIR / 'triplets'
for d in [OUT_DIR, FIG_DIR, TRIP_DIR]:
    d.mkdir(parents=True, exist_ok=True)

FOLD_DIRS: Dict[int, Dict[str, Path]] = {
    fold: {
        'images': BASE_DIR / f'fold_{fold}' / 'images' / f'fold_{fold}' / 'images.npy',
        'masks' : BASE_DIR / f'fold_{fold}' / 'masks'  / f'fold_{fold}' / 'masks.npy',
    }
    for fold in [1, 2, 3]
}

# ── Class metadata ────────────────────────────────────────────────────────────
# Short keys used for column names (no spaces / punctuation)
CLASS_KEYS: List[str] = [
    'Neoplastic', 'NonneoplasticEpi', 'Inflammatory', 'Connective', 'Dead'
]
# Display labels for plot axes / legends
CLASS_DISPLAY: List[str] = [
    'Neoplastic', 'Non-neo. Epi.', 'Inflammatory', 'Connective', 'Dead'
]
NUM_CLASSES = 5

# ── Hyperparameters ───────────────────────────────────────────────────────────
IMG_SIZE   = 256
THRESHOLD  = 0.5                    # sigmoid → binary prediction
IOU_THRS   = [0.25, 0.40, 0.50]    # CAM binarisation thresholds
N_PERMUTE  = 1000                   # permutation iterations
NOISE_THR  = 0.25                   # CAM overlay noise floor
ALPHA_SCALE = 0.85                  # overlay opacity

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# ── FPN layer config ──────────────────────────────────────────────────────────
# (layer_dotpath, blend_weight)
FPN_CFG: List[Tuple[str, float]] = [
    ('backbone.features.denseblock1', 0.20),
    ('backbone.features.denseblock2', 0.35),
    ('backbone.features.denseblock4', 0.45),
]
STD_CFG: List[Tuple[str, float]] = [
    ('backbone.features.denseblock4', 1.00),
]

# ── 2×2 method registry ───────────────────────────────────────────────────────
# key  →  (label, color, marker, use_pyramid, use_layercam)
METHOD_META: Dict[str, Tuple] = {
    'std_gradcam'  : ('Standard GradCAM',   '#90CAF9', 'o', False, False),
    'fpn_gradcam'  : ('FPN-GradCAM',        '#EF9A9A', 's', True,  False),
    'std_layercam' : ('Standard LayerCAM',  '#A5D6A7', '^', False, True ),
    'fpn_layercam' : ('FPN-LayerCAM',       '#FFE082', 'D', True,  True ),
}
METHODS = list(METHOD_META.keys())

def mlabel(k):  return METHOD_META[k][0]
def mcolor(k):  return METHOD_META[k][1]
def mmarker(k): return METHOD_META[k][2]

# ── Class colours for overlay ─────────────────────────────────────────────────
CLASS_COLORS_HEX = ['#E53935', '#43A047', '#1E88E5', '#FFB300', '#8E24AA']
CLASS_COLORS_F32: List[Tuple[float, float, float]] = [
    (int(h[1:3], 16) / 255, int(h[3:5], 16) / 255, int(h[5:7], 16) / 255)
    for h in CLASS_COLORS_HEX
]

print(f'Output directory : {OUT_DIR}')
print(f'Methods          : {METHODS}')

## 3. Model

In [ ]:
class DenseNet169MultiLabel(nn.Module):
    """DenseNet169 backbone + custom multi-label head (matches training notebook)."""

    def __init__(self, num_classes: int = 5, dropout_p: float = 0.3) -> None:
        super().__init__()
        backbone = models.densenet169(weights=None)
        in_features = backbone.classifier.in_features  # 1664
        backbone.classifier = nn.Identity()
        self.backbone = backbone
        self.classifier = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout_p),
            nn.Linear(512, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.backbone(x))


model = DenseNet169MultiLabel(num_classes=NUM_CLASSES)
ckpt  = torch.load(MODEL_PATH, map_location=DEVICE)
model.load_state_dict(ckpt['model_state'])
model.eval().to(DEVICE)
print(f"Checkpoint loaded — epoch {ckpt['epoch']}, val AUC {ckpt['val_auc']:.4f}")

## 4. CAM Engines — Unified 2×2 Implementation

`CAMEngine` is parameterised by two booleans:
- `use_pyramid` — `False` → single `denseblock4`; `True` → three-level FPN
- `use_layercam` — `False` → GradCAM (GAP); `True` → LayerCAM (pixel-wise)

All four methods share the same hook infrastructure and differ only in the
**four lines** inside `_cam_from_acts_grads`.

**Key engineering constraint:** each engine instance registers *its own* hooks
on the *same* `model` object.  Engines are used **sequentially** (one engine
per `generate()` call on a fresh input tensor), so concurrent hook writes
are never an issue.  Hooks are removed via `remove_hooks()` after all
processing is complete to prevent memory accumulation.

In [ ]:
class CAMEngine:
    """
    Unified CAM engine for all four methods in the 2×2 factorial design.

    Parameters
    ----------
    model        : trained DenseNet169MultiLabel (shared, eval mode)
    layer_cfg    : list of (dotpath, blend_weight) — one entry for standard,
                   three entries for FPN
    use_layercam : if True, use pixel-wise gradient (LayerCAM);
                   if False, use GAP of gradient (GradCAM)
    sharpen      : apply joint bilateral filter on the fused heatmap
                   (only applied when len(layer_cfg) > 1, i.e. FPN)
    """

    def __init__(
        self,
        model        : nn.Module,
        layer_cfg    : List[Tuple[str, float]],
        use_layercam : bool = False,
        sharpen      : bool = True,
    ) -> None:
        self.model        = model
        self.layer_cfg    = layer_cfg
        self.use_layercam = use_layercam
        # Sharpening only meaningful for multi-scale FPN output
        self.sharpen      = sharpen and (len(layer_cfg) > 1)

        self._handles: List = []
        # Per-layer activation / gradient store
        self._store: Dict[str, Dict[str, Optional[torch.Tensor]]] = {
            name: {'act': None, 'grad': None}
            for name, _ in layer_cfg
        }
        self._register_hooks()

    # ── Internal helpers ──────────────────────────────────────────────────────

    def _sub(self, dotpath: str) -> nn.Module:
        """Navigate model by dotted attribute path."""
        m = self.model
        for k in dotpath.split('.'):
            m = getattr(m, k)
        return m

    def _register_hooks(self) -> None:
        for name, _ in self.layer_cfg:
            mod = self._sub(name)
            n   = name  # capture by value in closure
            self._handles.append(
                mod.register_forward_hook(
                    lambda m, i, o, n=n:
                        self._store[n].__setitem__('act', o.detach())
                )
            )
            self._handles.append(
                mod.register_full_backward_hook(
                    lambda m, gi, go, n=n:
                        self._store[n].__setitem__('grad', go[0].detach())
                )
            )

    def remove_hooks(self) -> None:
        """Must be called after all processing to avoid accumulating hooks."""
        for h in self._handles:
            h.remove()
        self._handles.clear()

    # ── Core computation ──────────────────────────────────────────────────────

    def _cam_from_acts_grads(
        self,
        act : torch.Tensor,   # (1, C, h, w)
        grad: torch.Tensor,   # (1, C, h, w)
        hw  : Tuple[int, int],
    ) -> np.ndarray:           # (H, W) float32 in [0, 1]
        """
        The *only* difference between GradCAM and LayerCAM lives here.

        GradCAM  : alpha_k = GAP(grad)  →  (1, C, 1, 1) scalar weight
        LayerCAM : weights = ReLU(grad) →  (1, C, h, w) spatial map
        """
        if self.use_layercam:
            # LayerCAM: inner ReLU discards spatially-negative gradient
            # positions — only pixels where the class evidence is locally
            # positive contribute to the heatmap at that spatial location.
            # No GAP means local gradient peaks over individual nuclei survive.
            cam = F.relu(
                (F.relu(grad) * act).sum(dim=1, keepdim=True)   # (1,1,h,w)
            )
        else:
            # GradCAM: global average pool → one importance scalar per channel
            alpha = grad.mean(dim=(2, 3), keepdim=True)          # (1,C,1,1)
            cam   = F.relu(
                (alpha * act).sum(dim=1, keepdim=True)           # (1,1,h,w)
            )

        arr = cam.squeeze().cpu().numpy()         # (h, w)
        arr = np.maximum(arr, 0)

        # Upsample: cubic for FPN intermediate levels, bilinear for deep-only
        interp = cv2.INTER_CUBIC if self.sharpen else cv2.INTER_LINEAR
        arr    = cv2.resize(arr, (hw[1], hw[0]), interpolation=interp)

        if arr.max() > 1e-8:
            arr /= arr.max()
        return arr.astype(np.float32)

    @staticmethod
    def _bilateral_sharpen(
        cam  : np.ndarray,   # (H, W) float32 [0, 1]
        guide: np.ndarray,   # (H, W, 3) uint8 RGB — original image
    ) -> np.ndarray:
        """
        Joint bilateral filter: guide image prevents smoothing across
        intensity edges (nucleus boundaries), snapping the heatmap contour
        to actual nuclear membrane transitions in the H&E image.
        """
        gray = cv2.cvtColor(guide, cv2.COLOR_RGB2GRAY)
        u8   = (cam * 255).astype(np.uint8)
        if hasattr(cv2, 'ximgproc') and hasattr(cv2.ximgproc, 'jointBilateralFilter'):
            sharpened = cv2.ximgproc.jointBilateralFilter(
                gray, u8, d=9, sigmaColor=75, sigmaSpace=75
            )
        else:
            # Fallback: standard bilateral (no joint guidance)
            sharpened = cv2.bilateralFilter(u8, d=9, sigmaColor=75, sigmaSpace=75)
        result = sharpened.astype(np.float32) / 255.0
        if result.max() > 1e-8:
            result /= result.max()
        return result

    # ── Public API ────────────────────────────────────────────────────────────

    def generate(
        self,
        tensor    : torch.Tensor,          # (1, 3, H, W) requires_grad, on DEVICE
        class_idx : int,
        guide     : Optional[np.ndarray] = None,  # (H, W, 3) uint8 for sharpening
    ) -> np.ndarray:                        # (H, W) float32 [0, 1]
        H, W = tensor.shape[2], tensor.shape[3]

        # Single backward pass fills all registered hooks simultaneously
        self.model.zero_grad()
        self.model(tensor)[0, class_idx].backward(retain_graph=False)

        # Collect per-level CAMs
        level_cams: List[np.ndarray] = []
        weights:    List[float]      = []

        for name, w in self.layer_cfg:
            a = self._store[name]['act']
            g = self._store[name]['grad']
            if a is None or g is None:
                continue
            level_cams.append(self._cam_from_acts_grads(a, g, (H, W)))
            weights.append(w)

        if not level_cams:
            return np.zeros((H, W), dtype=np.float32)

        # Weighted pyramid blend (trivially a no-op for single-level)
        total = sum(weights)
        fused = sum(w / total * c for w, c in zip(weights, level_cams))
        if fused.max() > 1e-8:
            fused /= fused.max()

        # Edge-aware sharpening for FPN variants
        if self.sharpen and guide is not None:
            guide_r = cv2.resize(guide, (W, H))
            fused   = self._bilateral_sharpen(fused, guide_r)

        return fused.astype(np.float32)


# ── Instantiate all four engines ─────────────────────────────────────────────
# Each engine registers its own hooks onto the shared model.
# Engines are used sequentially (never concurrently) so hook writes
# from different engines do not interfere.
engines: Dict[str, CAMEngine] = {
    'std_gradcam'  : CAMEngine(model, STD_CFG, use_layercam=False),
    'fpn_gradcam'  : CAMEngine(model, FPN_CFG, use_layercam=False),
    'std_layercam' : CAMEngine(model, STD_CFG, use_layercam=True),
    'fpn_layercam' : CAMEngine(model, FPN_CFG, use_layercam=True),
}

print('CAMEngine instantiated for all 4 methods:')
for k in METHODS:
    _, _, _, pyr, lyr = METHOD_META[k]
    print(f'  {mlabel(k):<30}  pyramid={pyr}  layercam={lyr}')

## 5. Shared Preprocessing & IoU Utilities

In [ ]:
_transform = T.Compose([
    T.ToPILImage(),
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


def preprocess(img_rgb: np.ndarray) -> torch.Tensor:
    """
    HWC uint8 RGB → (1, 3, H, W) float32 on DEVICE with requires_grad=True.
    requires_grad is necessary for gradient-based CAM methods.
    """
    return _transform(img_rgb).unsqueeze(0).to(DEVICE).requires_grad_(True)


@torch.no_grad()
def predict(img_rgb: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """Returns (binary_labels, sigmoid_probs) for a single image."""
    t      = _transform(img_rgb).unsqueeze(0).to(DEVICE)
    logits = model(t)
    probs  = torch.sigmoid(logits).squeeze().cpu().numpy()
    return (probs >= THRESHOLD).astype(np.uint8), probs


def compute_iou(
    cam    : np.ndarray,   # (H, W) float32 [0, 1]
    gt_ch  : np.ndarray,   # (H, W) float32 — raw mask channel
    thr    : float,
) -> float:
    """
    IoU between binarised CAM and ground-truth mask channel.
    Returns NaN when the GT channel has no positive pixels
    (class absent) so it can be safely excluded from means.
    """
    gt_bin = gt_ch > 0
    if not gt_bin.any():
        return float('nan')
    pred  = cam >= thr
    inter = (pred & gt_bin).sum()
    union = (pred | gt_bin).sum()
    return float(inter) / float(union) if union > 0 else float('nan')


def resize_mask_ch(
    mask_6ch  : np.ndarray,   # (H, W, 6)
    class_idx : int,
    hw        : Tuple[int, int] = (IMG_SIZE, IMG_SIZE),
) -> np.ndarray:               # (H, W) float32
    ch = mask_6ch[..., class_idx].astype(np.float32)
    if ch.shape != hw:
        ch = cv2.resize(ch, (hw[1], hw[0]), interpolation=cv2.INTER_NEAREST)
    return ch


def render_gt_mask(
    mask_6ch: np.ndarray,
    hw: Tuple[int, int] = (IMG_SIZE, IMG_SIZE),
) -> np.ndarray:                # (H, W, 3) uint8
    H, W  = hw
    canvas = np.zeros((H, W, 3), dtype=np.float32)
    for c_idx, color in enumerate(CLASS_COLORS_F32):
        ch = resize_mask_ch(mask_6ch, c_idx, hw)
        canvas[ch > 0] = color
    return (canvas * 255).clip(0, 255).astype(np.uint8)


def build_cam_overlay(
    cams        : Dict[int, np.ndarray],   # {class_idx: (H,W) float32}
    active_idxs : List[int],
    orig_resized: np.ndarray,              # (H, W, 3) uint8
) -> np.ndarray:                           # (H, W, 3) uint8
    """
    Max-blend coloured class heatmaps over the original image.
    Pixel colour comes from the class with the highest CAM value at that pixel.
    """
    H, W   = IMG_SIZE, IMG_SIZE
    comp_a = np.zeros((H, W), dtype=np.float32)   # running max alpha
    comp_c = np.zeros((H, W, 3), dtype=np.float32) # running colour

    for c in active_idxs:
        cam   = cams.get(c, np.zeros((H, W), dtype=np.float32))
        clean = np.where(cam >= NOISE_THR, cam, 0.0)
        alpha = clean * ALPHA_SCALE
        mask  = alpha > comp_a
        comp_a[mask] = alpha[mask]
        r, g, b      = CLASS_COLORS_F32[c]
        comp_c[mask, 0] = clean[mask] * r
        comp_c[mask, 1] = clean[mask] * g
        comp_c[mask, 2] = clean[mask] * b

    bg  = orig_resized.astype(np.float32) / 255.0
    a   = comp_a[..., None]
    out = (comp_c * a + bg * (1 - a)).clip(0, 1)
    return (out * 255).astype(np.uint8)


print('Preprocessing and IoU utilities defined.')

## 6. Dataset

In [ ]:
class PanNukeFullDataset(Dataset):
    """
    Loads all three folds into memory.
    Images stored as uint8; masks as float16 to halve RAM (≈3.5 GB for full PanNuke).
    Labels derived from masks at construction time — no disk access per sample.
    """

    def __init__(self, fold_dirs: Dict[int, Dict[str, Path]]) -> None:
        imgs_list, masks_list, fold_ids, local_idxs = [], [], [], []

        for fold_id in [1, 2, 3]:
            print(f'  Loading fold {fold_id}...')
            imgs  = np.load(fold_dirs[fold_id]['images'], mmap_mode='r')
            masks = np.load(fold_dirs[fold_id]['masks'],  mmap_mode='r')
            n     = len(imgs)
            imgs_list.append(np.ascontiguousarray(imgs,  dtype=np.uint8))
            masks_list.append(np.ascontiguousarray(masks, dtype=np.float16))
            fold_ids.extend([fold_id] * n)
            local_idxs.extend(range(n))
            del imgs, masks
            gc.collect()

        self.images     = np.concatenate(imgs_list,  axis=0)   # (N, H, W, 3)
        self.masks      = np.concatenate(masks_list, axis=0)   # (N, H, W, 6)
        self.fold_ids   = np.array(fold_ids,   dtype=np.uint8)
        self.local_idxs = np.array(local_idxs, dtype=np.int32)

        # Multi-hot labels: class present iff any mask pixel is positive
        counts       = self.masks[..., :5].astype(np.float32).sum(axis=(1, 2))
        self.labels  = (counts > 0).astype(np.uint8)   # (N, 5)

        print(f'  Total: {len(self.images):,} images')
        print(f'  Label cardinality distribution:')
        cards = self.labels.sum(axis=1)
        for n in sorted(np.unique(cards)):
            print(f'    n={n}: {(cards==n).sum():,}')

    def __len__(self) -> int:
        return len(self.images)

    def __getitem__(self, idx: int):
        # masks: float16 → float32 at access time
        return (
            self.images[idx],                             # (H, W, 3) uint8
            self.masks[idx].astype(np.float32),           # (H, W, 6) float32
            self.labels[idx],                             # (5,) uint8
            int(self.fold_ids[idx]),
            int(self.local_idxs[idx]),
        )


print('Loading PanNuke dataset (all three folds)...')
full_ds = PanNukeFullDataset(FOLD_DIRS)
print(f'Dataset size: {len(full_ds):,} images')

## 7. Load Test Images, Predictions & GT Masks

In [ ]:
with open(TEST_JSON) as f:
    test_records = json.load(f)
N_TEST = len(test_records)
print(f'Test records loaded: {N_TEST}')

test_images_raw = np.array([
    np.array(Image.open(TEST_IMG_DIR / rec['filename']).convert('RGB'))
    for rec in test_records
])   # (N_TEST, H, W, 3) uint8
print(f'Test images: {test_images_raw.shape}')


def rebuild_test_masks(
    fold_dirs: Dict[int, Dict[str, Path]],
    n_test:    int = 50,
    seed:      int = SEED,
) -> np.ndarray:
    """
    Replicates the exact hold-out sampling from the training notebook:
    same RNG seed + per-fold counts → reproducible GT masks for the 50
    test images.
    """
    rng      = np.random.default_rng(seed)
    per_fold = n_test // 3
    extra    = n_test - 3 * per_fold           # fold 3 gets the remainder
    out      = []
    for fold_id, n in [(1, per_fold), (2, per_fold), (3, per_fold + extra)]:
        m_all = np.load(fold_dirs[fold_id]['masks'], mmap_mode='r')
        idx   = rng.choice(len(m_all), size=n, replace=False)
        out.append(np.asarray(m_all[idx], dtype=np.float32))
        del m_all
        gc.collect()
    return np.concatenate(out, axis=0)   # (n_test, H, W, 6)


print('Rebuilding test GT masks...')
test_masks = rebuild_test_masks(FOLD_DIRS)
print(f'Test masks: {test_masks.shape}')

## 8. Triplet Figure Renderer

In [ ]:
def save_triplet_figure(
    img_idx     : int,
    orig        : np.ndarray,          # (H, W, 3) uint8 resized
    gt_mask_rgb : np.ndarray,          # (H, W, 3) uint8
    overlay     : np.ndarray,          # (H, W, 3) uint8
    active_idxs : List[int],           # predicted-positive class indices
    probs       : List[float],         # sigmoid probs for all 5 classes
    gt_labels   : List[int],           # ground-truth binary labels
    iou_at_50   : Dict[int, float],    # {class_idx: IoU@0.50}
    method_label: str,
    save_path   : Path,
) -> None:
    fig = plt.figure(figsize=(16, 5.6), facecolor='#0d0d0d')
    gs  = fig.add_gridspec(1, 3, wspace=0.04)

    panels = [
        (orig,        'Original Image'),
        (gt_mask_rgb, 'Ground-Truth Mask'),
        (overlay,     f'{method_label} Overlay'),
    ]
    for col, (panel, title) in enumerate(panels):
        ax = fig.add_subplot(gs[0, col])
        ax.imshow(panel)
        ax.set_title(title, color='white', fontsize=10, fontweight='bold', pad=4)
        ax.axis('off')

    # Legend: one patch per predicted-positive class
    patches = []
    for c in active_idxs:
        marker = '✓' if bool(gt_labels[c]) else '✗'
        iou_v  = iou_at_50.get(c, float('nan'))
        iou_s  = f'{iou_v:.3f}' if not np.isnan(iou_v) else 'N/A'
        r, g, b = CLASS_COLORS_F32[c]
        patches.append(mpatches.Patch(
            facecolor=(r, g, b), edgecolor='white', linewidth=0.4,
            label=f'{marker} {CLASS_DISPLAY[c]:<18}  p={probs[c]:.2f}  IoU={iou_s}',
        ))

    if patches:
        fig.axes[2].legend(
            handles=patches, loc='lower right', fontsize=7.2,
            framealpha=0.82, facecolor='#111111', edgecolor='#555555',
            labelcolor='white', handlelength=1.0,
            borderpad=0.6, labelspacing=0.4,
        )

    fig.suptitle(
        f'Image #{img_idx:03d}  |  {method_label}  |  ✓ = TP   ✗ = FP',
        color='#dddddd', fontsize=10, fontweight='bold', y=1.02,
    )
    plt.savefig(save_path, dpi=150, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    plt.close(fig)


# Create triplet subdirectories
for k in METHODS:
    (TRIP_DIR / k).mkdir(exist_ok=True)

print('Triplet renderer ready.')

## 9. Run All 4 Methods on 50 Test Images

For each test image:
1. Generate a CAM heatmap per predicted-positive class for all 4 methods
2. Compute IoU at 3 thresholds
3. Save triplet figures
4. Accumulate results in `df_test`

> **Memory note (8 GB VRAM):** All four CAMs for one class are computed
> sequentially — one backward pass per engine — then the tensors are
> immediately freed. Peak VRAM usage is one image × one backward pass at a time.

In [ ]:
test_rows: List[Dict] = []

print(f'Processing {N_TEST} test images × {len(METHODS)} methods...\n')

for img_idx, rec in enumerate(tqdm(test_records, desc='Test images')):

    # ── Ground truth / predictions from saved JSON ────────────────────────────
    gt_lbl   = [int(rec['true_labels'][k])  for k in CLASS_KEYS]
    probs    = [float(rec['pred_probs'][k])  for k in CLASS_KEYS]
    pred_lbl = [int(rec['pred_labels'][k])  for k in CLASS_KEYS]
    active   = [i for i, p in enumerate(pred_lbl) if p == 1]

    orig_rgb     = test_images_raw[img_idx]                     # (H, W, 3)
    mask_6ch     = test_masks[img_idx]                          # (H, W, 6)
    orig_resized = cv2.resize(orig_rgb, (IMG_SIZE, IMG_SIZE))   # (256, 256, 3)
    gt_mask_rgb  = render_gt_mask(mask_6ch)                     # (256, 256, 3)

    row: Dict = {
        'image_idx'  : img_idx,
        'filename'   : rec['filename'],
        'cardinality': int(sum(gt_lbl)),
    }
    for j, ck in enumerate(CLASS_KEYS):
        row[f'gt_{ck}']   = gt_lbl[j]
        row[f'pred_{ck}'] = pred_lbl[j]
        row[f'prob_{ck}'] = round(probs[j], 4)

    # ── Generate CAMs: 4 methods × active classes ─────────────────────────────
    # all_cams[method_key][class_idx] = (256, 256) float32
    all_cams: Dict[str, Dict[int, np.ndarray]] = {m: {} for m in METHODS}

    for c_idx in active:
        for mkey, engine in engines.items():
            t   = preprocess(orig_rgb)   # fresh tensor with requires_grad each time
            cam = engine.generate(t, c_idx, guide=orig_resized)
            all_cams[mkey][c_idx] = cam
            del t

        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()

    # ── IoU per class × method × threshold ───────────────────────────────────
    for j, ck in enumerate(CLASS_KEYS):
        gt_ch = resize_mask_ch(mask_6ch, j)
        for mkey in METHODS:
            cam = all_cams[mkey].get(j, np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.float32))
            for thr in IOU_THRS:
                row[f'{mkey}_iou_{thr:.2f}_{ck}'] = round(compute_iou(cam, gt_ch, thr), 4)
            # Heatmap coverage (fraction of pixels above noise floor)
            row[f'{mkey}_area_{ck}'] = round(float((cam >= NOISE_THR).mean()), 4)

    test_rows.append(row)

    # ── Save triplet figures for all 4 methods ────────────────────────────────
    for mkey in METHODS:
        overlay = (
            build_cam_overlay(all_cams[mkey], active, orig_resized)
            if active else orig_resized.copy()
        )
        iou_50 = {
            c: compute_iou(
                all_cams[mkey].get(c, np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.float32)),
                resize_mask_ch(mask_6ch, c), 0.50,
            )
            for c in active
        }
        save_triplet_figure(
            img_idx=img_idx,
            orig=orig_resized,
            gt_mask_rgb=gt_mask_rgb,
            overlay=overlay,
            active_idxs=active,
            probs=probs,
            gt_labels=gt_lbl,
            iou_at_50=iou_50,
            method_label=mlabel(mkey),
            save_path=TRIP_DIR / mkey / f'{mkey}_{img_idx:03d}.png',
        )


df_test = pd.DataFrame(test_rows)
df_test.to_csv(OUT_DIR / 'results_50_test.csv', index=False)
print(f'\nTest CSV saved ({df_test.shape[0]} rows × {df_test.shape[1]} cols)')

## 10. Figure 1 — Per-Class IoU Bar Chart (3 thresholds × 4 methods)

In [ ]:
fig, axes = plt.subplots(1, len(IOU_THRS), figsize=(21, 6), sharey=False)
fig.suptitle(
    '2×2 Factorial CAM Comparison — Mean IoU on 50 Test Images\n'
    'Bars grouped by class; 4 methods per group at each threshold',
    fontsize=12, fontweight='bold',
)
x       = np.arange(NUM_CLASSES)
n_m     = len(METHODS)
w       = 0.18
offsets = np.linspace(-(n_m - 1) / 2 * w, (n_m - 1) / 2 * w, n_m)

for ax, thr in zip(axes, IOU_THRS):
    for mkey, off in zip(METHODS, offsets):
        means = [
            df_test[f'{mkey}_iou_{thr:.2f}_{ck}'].dropna().mean()
            for ck in CLASS_KEYS
        ]
        bars = ax.bar(
            x + off, means, w,
            label=mlabel(mkey), color=mcolor(mkey),
            edgecolor='white', linewidth=0.4, alpha=0.92,
        )
        for b, v in zip(bars, means):
            if not np.isnan(v):
                ax.text(
                    b.get_x() + b.get_width() / 2, b.get_height() + 0.003,
                    f'{v:.3f}', ha='center', va='bottom',
                    fontsize=5.2, rotation=90, color='#333333',
                )

    ax.set_xticks(x)
    ax.set_xticklabels(CLASS_DISPLAY, rotation=20, ha='right', fontsize=8)
    ax.set_title(f'IoU @ {thr:.2f}', fontsize=10, fontweight='bold')
    ax.set_ylabel('Mean IoU', fontsize=9)
    ax.set_ylim(0, 0.36)
    ax.legend(fontsize=7.5, loc='upper right', framealpha=0.6)
    ax.grid(axis='y', alpha=0.25, linestyle=':')

plt.tight_layout()
for ext in ['pdf', 'png']:
    plt.savefig(FIG_DIR / f'fig1_test_iou_4methods.{ext}', bbox_inches='tight', dpi=300)
plt.show()
print('Figure 1 saved.')

## 11. Figure 2 — IoU@0.50 Heatmap & Macro Summary Table

In [ ]:
# Build macro-average summary: method × threshold
summary_rows = []
for mkey in METHODS:
    for thr in IOU_THRS:
        class_means = [
            df_test[f'{mkey}_iou_{thr:.2f}_{ck}'].dropna().mean()
            for ck in CLASS_KEYS
        ]
        summary_rows.append({
            'method'    : mlabel(mkey),
            'threshold' : thr,
            'macro_iou' : round(float(np.nanmean(class_means)), 4),
            **{f'iou_{ck}': round(float(v), 4) for ck, v in zip(CLASS_KEYS, class_means)},
        })

df_summary = pd.DataFrame(summary_rows)
df_summary.to_csv(OUT_DIR / 'method_iou_summary.csv', index=False)

print('IoU@0.50 Macro Summary:')
print(df_summary[df_summary.threshold == 0.50][['method','macro_iou'] +
      [f'iou_{ck}' for ck in CLASS_KEYS]].to_string(index=False))

# Heatmap
df50       = df_summary[df_summary.threshold == 0.50].set_index('method')
hm_data    = df50[[f'iou_{ck}' for ck in CLASS_KEYS]].astype(float)
hm_data.columns = CLASS_DISPLAY

fig, axes = plt.subplots(1, 2, figsize=(18, 5),
                          gridspec_kw={'width_ratios': [2.2, 1]})
fig.suptitle('IoU@0.50 — 4 Methods × 5 Classes (50 test images)',
             fontsize=12, fontweight='bold')

sns.heatmap(
    hm_data, ax=axes[0],
    cmap='YlOrRd', annot=True, fmt='.4f', annot_kws={'size': 10},
    linewidths=0.5, linecolor='#cccccc',
    cbar_kws={'label': 'Mean IoU@0.50', 'shrink': 0.82},
)
axes[0].set_title('Per-class IoU@0.50', fontsize=10)
axes[0].tick_params(axis='x', rotation=20, labelsize=9)
axes[0].tick_params(axis='y', rotation=0,  labelsize=9)

# Macro bar chart on the right panel
macro_vals = df50['macro_iou'].values
axes[1].barh(
    [mlabel(m) for m in METHODS],
    macro_vals,
    color=[mcolor(m) for m in METHODS],
    edgecolor='white', linewidth=0.4, alpha=0.9,
)
for i, v in enumerate(macro_vals):
    axes[1].text(v + 0.001, i, f'{v:.4f}', va='center', fontsize=9)
axes[1].set_xlabel('Macro IoU@0.50', fontsize=9)
axes[1].set_title('Macro-averaged', fontsize=10)
axes[1].set_xlim(0, max(macro_vals) * 1.25)
axes[1].grid(axis='x', alpha=0.25, linestyle=':')

plt.tight_layout()
for ext in ['pdf', 'png']:
    plt.savefig(FIG_DIR / f'fig2_iou_heatmap_macro.{ext}', bbox_inches='tight', dpi=300)
plt.show()
print('Figure 2 saved.')

## 12. Figure 3 — Axis Decomposition: Gradient Treatment vs Resolution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(
    'Factorial Axis Decomposition at IoU@0.50 (50 test images)\n'
    'Left: effect of gradient treatment  |  Right: effect of resolution strategy',
    fontsize=11, fontweight='bold',
)
x, w = np.arange(NUM_CLASSES), 0.35

# Panel A: gradient treatment (single layer only)
ax = axes[0]
for i, (mkey, lbl) in enumerate([('std_gradcam',  'Standard GradCAM (GAP)'),
                                   ('std_layercam', 'Standard LayerCAM (pixel-wise)')]):
    means = [df_test[f'{mkey}_iou_0.50_{ck}'].dropna().mean() for ck in CLASS_KEYS]
    bars  = ax.bar(x + (i - 0.5) * w, means, w, label=lbl,
                   color=mcolor(mkey), edgecolor='white', linewidth=0.4, alpha=0.9)
    for b, v in zip(bars, means):
        if not np.isnan(v):
            ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.003,
                    f'{v:.3f}', ha='center', va='bottom', fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(CLASS_DISPLAY, rotation=18, ha='right', fontsize=9)
ax.set_title('Gradient Treatment\n(Single layer — denseblock4)', fontsize=10, fontweight='bold')
ax.set_ylabel('Mean IoU@0.50', fontsize=9)
ax.set_ylim(0, 0.30)
ax.legend(fontsize=9, framealpha=0.6)
ax.grid(axis='y', alpha=0.25, linestyle=':')

# Panel B: resolution strategy (GradCAM weighting only)
ax = axes[1]
for i, (mkey, lbl) in enumerate([('std_gradcam', 'Standard GradCAM (single layer)'),
                                   ('fpn_gradcam', 'FPN-GradCAM (3-level pyramid)')]):
    means = [df_test[f'{mkey}_iou_0.50_{ck}'].dropna().mean() for ck in CLASS_KEYS]
    bars  = ax.bar(x + (i - 0.5) * w, means, w, label=lbl,
                   color=mcolor(mkey), edgecolor='white', linewidth=0.4, alpha=0.9)
    for b, v in zip(bars, means):
        if not np.isnan(v):
            ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.003,
                    f'{v:.3f}', ha='center', va='bottom', fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(CLASS_DISPLAY, rotation=18, ha='right', fontsize=9)
ax.set_title('Resolution Strategy\n(GAP weighting — GradCAM family)', fontsize=10, fontweight='bold')
ax.set_ylabel('Mean IoU@0.50', fontsize=9)
ax.set_ylim(0, 0.30)
ax.legend(fontsize=9, framealpha=0.6)
ax.grid(axis='y', alpha=0.25, linestyle=':')

plt.tight_layout()
for ext in ['pdf', 'png']:
    plt.savefig(FIG_DIR / f'fig3_axis_decomposition.{ext}', bbox_inches='tight', dpi=300)
plt.show()
print('Figure 3 saved.')

## 13. Figure 4 — Visual Comparison Grid (4 images × 4 methods)

In [ ]:
N_PREVIEW = 4   # number of example images shown

fig, axes = plt.subplots(
    N_PREVIEW, N_PREVIEW + 1,
    figsize=(5.5 * (N_PREVIEW + 1), 5.5 * N_PREVIEW),
)
fig.patch.set_facecolor('#0d0d0d')
fig.suptitle(
    'Visual Comparison: GT Mask (col 1) then Standard GradCAM / FPN-GradCAM /'
    ' Standard LayerCAM / FPN-LayerCAM',
    color='white', fontsize=12, fontweight='bold', y=1.004,
)

col_titles = ['GT Mask'] + [mlabel(m) for m in METHODS]

for row_i in range(N_PREVIEW):
    # Column 0: ground-truth mask
    axes[row_i, 0].imshow(render_gt_mask(test_masks[row_i]))
    axes[row_i, 0].axis('off')
    if row_i == 0:
        axes[row_i, 0].set_title(col_titles[0], color='white',
                                  fontsize=9, fontweight='bold', pad=4)

    # Columns 1–4: CAM overlay panels (extracted from saved triplets)
    for col_i, mkey in enumerate(METHODS):
        ax   = axes[row_i, col_i + 1]
        fpath = TRIP_DIR / mkey / f'{mkey}_{row_i:03d}.png'
        if fpath.exists():
            full = np.array(Image.open(fpath))
            # Triplet is 3 equal panels — right-most is the overlay
            pw = full.shape[1] // 3
            ax.imshow(full[:, 2 * pw:])
        else:
            ax.text(0.5, 0.5, 'N/A', ha='center', color='white',
                    transform=ax.transAxes)
        ax.axis('off')
        if row_i == 0:
            ax.set_title(col_titles[col_i + 1], color='white',
                         fontsize=9, fontweight='bold', pad=4)

plt.tight_layout(pad=0.4)
plt.savefig(
    FIG_DIR / 'fig4_visual_grid.png',
    dpi=120, bbox_inches='tight', facecolor=fig.get_facecolor(),
)
plt.show()
print('Figure 4 saved (PNG only — large raster).')

## 14. Full-Dataset Inference — All 4 Methods

Process every correctly-classified image in the full PanNuke dataset.
The exact-match filter keeps images where `predicted_labels == gt_labels`
so that IoU differences between cardinality groups are attributable to
CAM localisation failure rather than classification errors.

In [ ]:
full_rows : List[Dict] = []
n_skipped : int        = 0

print(f'Full-dataset pass: {len(full_ds):,} images × {len(METHODS)} methods...')

for gidx in tqdm(range(len(full_ds)), desc='Full dataset'):
    img_raw, mask_6ch, gt_labels, fold_id, local_idx = full_ds[gidx]
    cardinality = int(gt_labels.sum())

    # Exact-match filter (no gradient needed)
    pred_labels, probs = predict(img_raw)
    if not np.array_equal(gt_labels, pred_labels):
        n_skipped += 1
        continue

    active = [i for i, p in enumerate(pred_labels) if p == 1]
    if not active:
        n_skipped += 1
        continue

    img_resized = cv2.resize(img_raw, (IMG_SIZE, IMG_SIZE))

    row: Dict = {
        'global_idx' : gidx,
        'fold_id'    : int(fold_id),
        'local_idx'  : int(local_idx),
        'cardinality': cardinality,
        'label_str'  : '|'.join(CLASS_KEYS[i] for i in range(NUM_CLASSES) if gt_labels[i]),
    }
    for j, ck in enumerate(CLASS_KEYS):
        row[f'gt_{ck}']   = int(gt_labels[j])
        row[f'pred_{ck}'] = int(pred_labels[j])
        row[f'prob_{ck}'] = round(float(probs[j]), 4)

    # CAMs and IoU for each active class × each method
    for c_idx in active:
        ck    = CLASS_KEYS[c_idx]
        gt_ch = resize_mask_ch(mask_6ch, c_idx)

        for mkey, engine in engines.items():
            t   = preprocess(img_raw)
            cam = engine.generate(t, c_idx, guide=img_resized)
            del t

            for thr in IOU_THRS:
                row[f'{mkey}_iou_{thr:.2f}_{ck}'] = round(
                    compute_iou(cam, gt_ch, thr), 4
                )
            row[f'{mkey}_area_{ck}'] = round(float((cam >= NOISE_THR).mean()), 4)

    full_rows.append(row)

    if DEVICE.type == 'cuda' and gidx % 200 == 0:
        torch.cuda.empty_cache()


# Remove hooks now that all CAM generation is complete
for eng in engines.values():
    eng.remove_hooks()

df_full = pd.DataFrame(full_rows)

# Ensure all IoU columns exist (fill NaN for non-active classes)
for mkey in METHODS:
    for ck in CLASS_KEYS:
        for thr in IOU_THRS:
            col = f'{mkey}_iou_{thr:.2f}_{ck}'
            if col not in df_full.columns:
                df_full[col] = np.nan

df_full.to_csv(OUT_DIR / 'results_full_dataset.csv', index=False)
print(f'\nFull dataset CSV: {len(df_full):,} retained, {n_skipped:,} skipped')
print(f'Shape: {df_full.shape}')
print('Cardinality distribution:')
print(df_full['cardinality'].value_counts().sort_index())

## 15. Figure 5 — IoU vs Label Cardinality (Monotonicity Test)

In [ ]:
def pooled_iou_by_cardinality(
    df: pd.DataFrame, mkey: str, thr: float = 0.50
) -> pd.DataFrame:
    """Mean IoU pooled over all classes, grouped by label cardinality."""
    thr_s = f'{thr:.2f}'
    rows  = []
    for card in sorted(df['cardinality'].unique()):
        sub  = df[df['cardinality'] == card]
        vals = np.concatenate([
            sub[f'{mkey}_iou_{thr_s}_{ck}'].dropna().values
            for ck in CLASS_KEYS
        ])
        rows.append({
            'cardinality': card,
            'mean'       : float(np.nanmean(vals)),
            'sem'        : float(np.nanstd(vals) / max(np.sqrt(len(vals)), 1)),
            'n_obs'      : len(vals),
        })
    return pd.DataFrame(rows)


fig, ax = plt.subplots(figsize=(10, 6))
fig.suptitle(
    'IoU@0.50 vs Label Cardinality — All 4 Methods\n'
    '(correctly-classified images, all 5 classes pooled; error bars = 95% CI)',
    fontsize=12, fontweight='bold',
)

spearman_rows = []
for mkey in METHODS:
    cd       = pooled_iou_by_cardinality(df_full, mkey)
    rho, p   = stats.spearmanr(cd['cardinality'], cd['mean'])
    spearman_rows.append({'method': mlabel(mkey), 'rho': rho, 'p': p})
    ax.errorbar(
        cd['cardinality'], cd['mean'],
        yerr=cd['sem'] * 1.96,
        fmt=f"{mmarker(mkey)}-",
        color=mcolor(mkey), linewidth=2.5, markersize=9, capsize=5,
        label=f"{mlabel(mkey)}  (ρ={rho:+.3f}, p={p:.4f})",
    )

ax.set_xlabel('Label Cardinality (n)', fontsize=11)
ax.set_ylabel('Mean IoU@0.50 ± 95% CI', fontsize=11)
ax.set_xticks(sorted(df_full['cardinality'].unique()))
ax.legend(fontsize=9, loc='upper right', framealpha=0.7)
ax.grid(alpha=0.25, linestyle=':')

plt.tight_layout()
for ext in ['pdf', 'png']:
    plt.savefig(FIG_DIR / f'fig5_iou_vs_cardinality.{ext}', bbox_inches='tight', dpi=300)
plt.show()

df_spearman = pd.DataFrame(spearman_rows)
print('Spearman monotonicity test (cardinality vs IoU@0.50):')
print(df_spearman.to_string(index=False))
print('Figure 5 saved.')

## 16. Figure 6 — Per-Class n=1 vs n≥2 (Welch's t-test)

In [ ]:
drop_rows: List[Dict] = []

fig, axes = plt.subplots(1, NUM_CLASSES, figsize=(22, 6))
fig.suptitle(
    'Per-Class IoU@0.50: n=1 (solid) vs n≥2 (hatched) — 4 Methods\n'
    'Positive ΔIoU = degradation; Negative = improvement under co-occurrence',
    fontsize=11, fontweight='bold',
)

x_m = np.arange(len(METHODS))

for ax, ck, cdisp in zip(axes, CLASS_KEYS, CLASS_DISPLAY):
    for mkey in METHODS:
        col    = f'{mkey}_iou_0.50_{ck}'
        if col not in df_full.columns:
            continue
        single = df_full[df_full['cardinality'] == 1][col].dropna().values
        multi  = df_full[df_full['cardinality'] >= 2][col].dropna().values
        sm     = float(np.nanmean(single)) if len(single) >= 2 else np.nan
        mm     = float(np.nanmean(multi))  if len(multi)  >= 2 else np.nan
        drop   = sm - mm if not (np.isnan(sm) or np.isnan(mm)) else np.nan
        if len(single) >= 2 and len(multi) >= 2:
            t_stat, p_val = stats.ttest_ind(single, multi, equal_var=False)
        else:
            t_stat, p_val = np.nan, np.nan
        drop_rows.append({
            'method'    : mlabel(mkey), 'class': cdisp,
            'iou_n1'    : round(sm,   4) if not np.isnan(sm)   else np.nan,
            'iou_nmulti': round(mm,   4) if not np.isnan(mm)   else np.nan,
            'delta'     : round(drop, 4) if not np.isnan(drop) else np.nan,
            'p_val'     : round(p_val,4) if not np.isnan(p_val) else np.nan,
            'n_single'  : len(single), 'n_multi': len(multi),
        })

    # Bar chart for this class
    smeans = []
    mmeans = []
    for mkey in METHODS:
        col    = f'{mkey}_iou_0.50_{ck}'
        single = df_full[df_full['cardinality'] == 1][col].dropna().values if col in df_full.columns else np.array([])
        multi  = df_full[df_full['cardinality'] >= 2][col].dropna().values if col in df_full.columns else np.array([])
        smeans.append(float(np.nanmean(single)) if len(single) >= 2 else np.nan)
        mmeans.append(float(np.nanmean(multi))  if len(multi)  >= 2 else np.nan)

    colors_ = [mcolor(m) for m in METHODS]
    ax.bar(x_m - 0.2, smeans, 0.35, color=colors_, alpha=0.92,
           edgecolor='white', linewidth=0.4)
    ax.bar(x_m + 0.2, mmeans, 0.35, color=colors_, alpha=0.55,
           edgecolor='white', linewidth=0.4, hatch='///')

    ax.set_xticks(x_m)
    ax.set_xticklabels(
        [mlabel(m).replace(' ', '\n') for m in METHODS],
        fontsize=6.2,
    )
    ax.set_title(cdisp, fontsize=9, fontweight='bold')
    ax.set_ylabel('Mean IoU@0.50', fontsize=8)
    ax.grid(axis='y', alpha=0.25, linestyle=':')

# Legend in first panel only
axes[0].legend(
    handles=[
        Patch(facecolor='grey', alpha=0.92, label='n=1  (solid)'),
        Patch(facecolor='grey', alpha=0.55, hatch='///', label='n≥2  (hatched)'),
    ],
    fontsize=7.5, loc='upper right', framealpha=0.7,
)

plt.tight_layout()
for ext in ['pdf', 'png']:
    plt.savefig(FIG_DIR / f'fig6_per_class_n1_vs_nmulti.{ext}', bbox_inches='tight', dpi=300)
plt.show()

df_drop = pd.DataFrame(drop_rows)
df_drop.to_csv(OUT_DIR / 'per_class_n1_vs_nmulti.csv', index=False)
print('Figure 6 + per_class_n1_vs_nmulti.csv saved.')

## 17. Pairwise Interference Matrices — All 4 Methods

In [ ]:
def build_interference_matrix(
    df    : pd.DataFrame,
    mkey  : str,
    thr   : float = 0.50,
    min_n : int   = 5,
) -> Tuple[np.ndarray, pd.DataFrame]:
    """
    Compute the 5×5 pairwise interference matrix:
      ΔIoU(c | c') = mean IoU_c(c' absent) − mean IoU_c(c' present)

    Positive delta: class c is harder to localise when c' co-occurs.
    Negative delta: class c is *easier* to localise when c' co-occurs
                    (synergistic / negative interference).
    """
    thr_s  = f'{thr:.2f}'
    matrix = np.full((NUM_CLASSES, NUM_CLASSES), np.nan)
    detail : List[Dict] = []

    for c in range(NUM_CLASSES):
        ck     = CLASS_KEYS[c]
        iou_col = f'{mkey}_iou_{thr_s}_{ck}'
        if iou_col not in df.columns:
            continue
        # Images where class c is present AND correctly predicted
        base = df[(df[f'gt_{ck}'] == 1) & (df[f'pred_{ck}'] == 1)][[iou_col, *[f'gt_{CLASS_KEYS[cp]}' for cp in range(NUM_CLASSES)]]].copy()

        for cp in range(NUM_CLASSES):
            if c == cp:
                continue
            ck_p    = CLASS_KEYS[cp]
            absent  = base[base[f'gt_{ck_p}'] == 0][iou_col].dropna().values
            present = base[base[f'gt_{ck_p}'] == 1][iou_col].dropna().values

            if len(absent) < min_n or len(present) < min_n:
                continue

            delta         = float(np.nanmean(absent)) - float(np.nanmean(present))
            t_stat, p_val = stats.ttest_ind(absent, present, equal_var=False)
            matrix[c, cp] = delta

            detail.append({
                'method'     : mkey,
                'class_c'    : CLASS_DISPLAY[c],
                'class_cp'   : CLASS_DISPLAY[cp],
                'delta_iou'  : round(delta, 4),
                'iou_absent' : round(float(np.nanmean(absent)),  4),
                'iou_present': round(float(np.nanmean(present)), 4),
                'n_absent'   : len(absent),
                'n_present'  : len(present),
                't_stat'     : round(float(t_stat), 4),
                'p_val'      : round(float(p_val),  4),
                'significant': bool(p_val < 0.05),
                'direction'  : 'positive' if delta > 0 else 'negative',
            })

    return matrix, pd.DataFrame(detail)


int_matrices : Dict[str, np.ndarray] = {}
int_details  : List[pd.DataFrame]    = []

for mkey in METHODS:
    mat, det = build_interference_matrix(df_full, mkey)
    int_matrices[mkey] = mat
    int_details.append(det)

df_pairwise = pd.concat(int_details, ignore_index=True)
df_pairwise.to_csv(OUT_DIR / 'pairwise_interference_all_methods.csv', index=False)

print('Pairwise interference computed:')
for mkey in METHODS:
    sub     = df_pairwise[df_pairwise.method == mkey]
    n_sig   = sub['significant'].sum()
    n_pos   = ((sub['significant']) & (sub['delta_iou'] > 0)).sum()
    n_neg   = ((sub['significant']) & (sub['delta_iou'] < 0)).sum()
    print(f'  {mlabel(mkey):<30}: {n_sig}/{len(sub)} sig  '
          f'({n_pos} positive, {n_neg} negative interference)')

## 18. Figure 7 — 2×2 Interference Matrix Grid

In [ ]:
short = [d[:11] for d in CLASS_DISPLAY]

# Symmetric colour scale across all four matrices
vmax = np.nanmax(np.abs(np.concatenate([
    m[~np.isnan(m)].flatten() for m in int_matrices.values()
])))

fig, axes = plt.subplots(2, 2, figsize=(16, 13))
fig.suptitle(
    r'Pairwise Interference $\Delta$IoU$(c\,|\,c^{\prime})$ @ 0.50'
    r'  —  Row = measured class $c$,  Column = co-occurring $c^{\prime}$'  '\n'
    'Positive (red) = co-occurrence degrades localisation; '
    'Negative (blue) = co-occurrence improves localisation',
    fontsize=11, fontweight='bold',
)

layout = [(0, 0, 'std_gradcam'), (0, 1, 'fpn_gradcam'),
           (1, 0, 'std_layercam'), (1, 1, 'fpn_layercam')]

for ri, ci, mkey in layout:
    ax  = axes[ri][ci]
    mat = int_matrices[mkey]
    sns.heatmap(
        mat, ax=ax,
        mask=np.eye(NUM_CLASSES, dtype=bool),
        cmap='RdBu_r', center=0, vmin=-vmax, vmax=vmax,
        annot=True, fmt='.3f', annot_kws={'size': 9},
        xticklabels=short, yticklabels=short,
        linewidths=0.4, linecolor='#cccccc',
        cbar=(ci == 1),
        cbar_kws={'label': 'ΔIoU', 'shrink': 0.85} if ci == 1 else {},
    )
    ax.set_title(mlabel(mkey), fontsize=11, fontweight='bold')
    ax.tick_params(axis='x', rotation=30, labelsize=8)
    ax.tick_params(axis='y', rotation=0,  labelsize=8)
    ax.set_xlabel("Co-occurring class $c'$", fontsize=8)
    ax.set_ylabel('Measured class $c$',     fontsize=8)

# Design-axis labels in the margins
fig.text(0.01, 0.74, 'Single\nLayer',      ha='center', fontsize=10, fontweight='bold',
         color='#1565C0', rotation=90, va='center')
fig.text(0.01, 0.28, 'Feature\nPyramid',   ha='center', fontsize=10, fontweight='bold',
         color='#1565C0', rotation=90, va='center')
fig.text(0.28, 0.98, 'GAP Gradient (GradCAM)',        ha='center', fontsize=10,
         fontweight='bold', color='#1565C0')
fig.text(0.73, 0.98, 'Pixel-wise Gradient (LayerCAM)', ha='center', fontsize=10,
         fontweight='bold', color='#1565C0')

plt.tight_layout(rect=[0.04, 0, 1, 0.97])
for ext in ['pdf', 'png']:
    plt.savefig(FIG_DIR / f'fig7_interference_matrix_2x2.{ext}', bbox_inches='tight', dpi=300)
plt.show()
print('Figure 7 saved.')

## 19. Permutation Tests — All 4 Methods

In [ ]:
def run_permutation_test(
    df    : pd.DataFrame,
    mkey  : str,
    c_idx : int,
    cp_idx: int,
    thr   : float = 0.50,
    n_perm: int   = N_PERMUTE,
    rng   : np.random.Generator = None,
) -> Optional[Dict]:
    """One-sided permutation test: H1: observed delta > null distribution."""
    if rng is None:
        rng = np.random.default_rng(SEED)

    ck  = CLASS_KEYS[c_idx]
    ck_p = CLASS_KEYS[cp_idx]
    col = f'{mkey}_iou_{thr:.2f}_{ck}'
    if col not in df.columns:
        return None

    base = (
        df[(df[f'gt_{ck}'] == 1) & (df[f'pred_{ck}'] == 1)]
        [[col, f'gt_{ck_p}']].dropna()
    )
    if len(base) < 10:
        return None

    co_lbl = base[f'gt_{ck_p}'].values.copy()
    iou_v  = base[col].values

    absent  = iou_v[co_lbl == 0]
    present = iou_v[co_lbl == 1]
    if len(absent) < 3 or len(present) < 3:
        return None

    observed = float(np.nanmean(absent)) - float(np.nanmean(present))

    null_deltas = np.empty(n_perm, dtype=np.float64)
    for i in range(n_perm):
        perm   = rng.permutation(co_lbl)
        null_deltas[i] = (
            np.nanmean(iou_v[perm == 0]) - np.nanmean(iou_v[perm == 1])
        )

    p_emp = float((null_deltas >= observed).mean())

    return {
        'method'         : mkey,
        'class_c'        : CLASS_DISPLAY[c_idx],
        'class_cp'       : CLASS_DISPLAY[cp_idx],
        'observed_delta' : round(observed, 4),
        'null_mean'      : round(float(null_deltas.mean()), 4),
        'null_std'       : round(float(null_deltas.std()),  4),
        'p_empirical'    : round(p_emp, 4),
        'significant'    : bool(p_emp < 0.05),
        'n_absent'       : len(absent),
        'n_present'      : len(present),
    }


rng_perm  = np.random.default_rng(SEED)
perm_rows : List[Dict] = []

for mkey in tqdm(METHODS, desc='Permutation tests'):
    for c in range(NUM_CLASSES):
        for cp in range(NUM_CLASSES):
            if c == cp:
                continue
            res = run_permutation_test(df_full, mkey, c, cp, rng=rng_perm)
            if res is not None:
                perm_rows.append(res)

df_perm = pd.DataFrame(perm_rows)
df_perm.to_csv(OUT_DIR / 'permutation_results_all_methods.csv', index=False)

print('Permutation test results (significant pairs):')
for mkey in METHODS:
    sub = df_perm[df_perm.method == mkey]
    sig = sub['significant'].sum()
    print(f'  {mlabel(mkey):<30}: {sig}/{len(sub)} significant (p<0.05)')

## 20. Figure 8 — Permutation Test Null Distributions (Top Interfering Pair)

In [ ]:
# Identify the most interfering pair across all methods (highest observed delta)
top_row = df_perm.loc[df_perm['observed_delta'].idxmax()]
c_top   = CLASS_DISPLAY.index(top_row['class_c'])
cp_top  = CLASS_DISPLAY.index(top_row['class_cp'])

print(f'Most interfering pair: {top_row["class_c"]} | {top_row["class_cp"]}  '
      f'(delta={top_row["observed_delta"]:.3f})')

fig, axes = plt.subplots(1, len(METHODS), figsize=(22, 5), sharey=False)
fig.suptitle(
    f'Permutation Test (N={N_PERMUTE}) — Pair: '
    f'{top_row["class_c"]} | {top_row["class_cp"]}\n'
    f'Null distribution vs observed ΔIoU@0.50 for each method',
    fontsize=11, fontweight='bold',
)

rng2 = np.random.default_rng(SEED)
for ax, mkey in zip(axes, METHODS):
    ck  = CLASS_KEYS[c_top]
    ck_p = CLASS_KEYS[cp_top]
    col = f'{mkey}_iou_0.50_{ck}'

    if col not in df_full.columns:
        ax.text(0.5, 0.5, 'N/A', ha='center', transform=ax.transAxes)
        continue

    base = df_full[(df_full[f'gt_{ck}'] == 1) & (df_full[f'pred_{ck}'] == 1)][
        [col, f'gt_{ck_p}']].dropna()
    if len(base) < 10:
        ax.text(0.5, 0.5, 'Insufficient n', ha='center',
                va='center', transform=ax.transAxes)
        continue

    co_lbl   = base[f'gt_{ck_p}'].values
    iou_vals = base[col].values
    observed = float(np.nanmean(iou_vals[co_lbl == 0])) - float(np.nanmean(iou_vals[co_lbl == 1]))

    null = np.array([
        np.nanmean(iou_vals[rng2.permutation(co_lbl) == 0]) -
        np.nanmean(iou_vals[rng2.permutation(co_lbl) == 1])
        for _ in range(N_PERMUTE)
    ])
    p_emp = float((null >= observed).mean())

    ax.hist(null, bins=45, color=mcolor(mkey), edgecolor='white',
            alpha=0.85, linewidth=0.4)
    ax.axvline(observed,    color='#C62828', linewidth=2.5, linestyle='--',
               label=f'Obs. = {observed:.3f}')
    ax.axvline(null.mean(), color='#333333', linewidth=1.5, linestyle=':',
               label=f'Null μ = {null.mean():.3f}')
    ax.set_title(f'{mlabel(mkey)}\np_emp = {p_emp:.3f}',
                 fontsize=9, fontweight='bold')
    ax.set_xlabel('ΔIoU (permuted co-occurrence)', fontsize=8)
    ax.set_ylabel('Count', fontsize=8)
    ax.legend(fontsize=7.5, framealpha=0.7)
    ax.grid(alpha=0.2, linestyle=':')

plt.tight_layout()
for ext in ['pdf', 'png']:
    plt.savefig(FIG_DIR / f'fig8_permutation_null_dists.{ext}', bbox_inches='tight', dpi=300)
plt.show()
print('Figure 8 saved.')

## 21. Figure 9 — LayerCAM Robustness Gain vs GradCAM (Wilcoxon Test)

In [ ]:
# Pivot interference table: rows = (class_c, class_cp), cols = method
pivot = df_pairwise.pivot_table(
    index=['class_c', 'class_cp'],
    columns='method',
    values='delta_iou',
).reset_index()
pivot.columns.name = None

# Robustness gain = GradCAM_delta - LayerCAM_delta
# Positive gain → LayerCAM has lower interference for that pair
pivot['gain_single'] = pivot.get('std_gradcam', np.nan) - pivot.get('std_layercam', np.nan)
pivot['gain_fpn']    = pivot.get('fpn_gradcam', np.nan) - pivot.get('fpn_layercam', np.nan)
pivot.to_csv(OUT_DIR / 'robustness_layercam_vs_gradcam.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 7.5))
fig.suptitle(
    'LayerCAM Robustness Gain vs GradCAM  (ΔIoU_GradCAM − ΔIoU_LayerCAM)\n'
    'Positive (green) = LayerCAM less affected by interference for that pair',
    fontsize=11, fontweight='bold',
)

wilcoxon_rows = []

for ax, gain_col, label_a, label_b in [
    (axes[0], 'gain_single', 'std_gradcam',  'std_layercam'),
    (axes[1], 'gain_fpn',    'fpn_gradcam',  'fpn_layercam'),
]:
    sub = pivot.dropna(subset=[gain_col]).sort_values(gain_col)
    if sub.empty:
        ax.text(0.5, 0.5, 'No data', ha='center', transform=ax.transAxes)
        continue

    y_labels = [
        f"{r['class_c'][:10]}  →  {r['class_cp'][:10]}"
        for _, r in sub.iterrows()
    ]
    gains  = sub[gain_col].values
    colors = ['#388E3C' if g > 0 else '#C62828' for g in gains]

    ax.barh(range(len(gains)), gains, color=colors,
            alpha=0.85, edgecolor='white', linewidth=0.4)
    ax.set_yticks(range(len(y_labels)))
    ax.set_yticklabels(y_labels, fontsize=7.5)
    ax.axvline(0, color='black', linewidth=1)
    ax.set_xlabel('ΔIoU reduction (GradCAM − LayerCAM)', fontsize=9)
    ax.grid(axis='x', alpha=0.2, linestyle=':')

    # Wilcoxon signed-rank test: is LayerCAM systematically more robust?
    valid_a = pivot.dropna(subset=[label_a, label_b])
    if len(valid_a) >= 5:
        try:
            W, p_wil = stats.wilcoxon(
                valid_a[label_a].values,
                valid_a[label_b].values,
                alternative='greater',
            )
        except ValueError:  # all differences zero
            W, p_wil = np.nan, np.nan
    else:
        W, p_wil = np.nan, np.nan

    title = (
        f'{mlabel(label_a).split("-")[0].strip()} vs '
        f'{mlabel(label_b).split("-")[0].strip()}\n'
        f'Wilcoxon W={W:.0f}, p={p_wil:.4f}' if not np.isnan(W)
        else f'{mlabel(label_a)} vs {mlabel(label_b)}'
    )
    ax.set_title(title, fontsize=9, fontweight='bold')

    wilcoxon_rows.append({
        'comparison' : f'{mlabel(label_a)} > {mlabel(label_b)}',
        'W'          : W, 'p_wilcoxon': round(p_wil, 4) if not np.isnan(p_wil) else np.nan,
        'n_pairs'    : len(valid_a),
        'n_layercam_wins': int((gains > 0).sum()),
        'n_pairs_total'  : len(gains),
    })

plt.tight_layout()
for ext in ['pdf', 'png']:
    plt.savefig(FIG_DIR / f'fig9_layercam_robustness_gain.{ext}', bbox_inches='tight', dpi=300)
plt.show()

print('\nWilcoxon signed-rank results (H1: GradCAM_delta > LayerCAM_delta):')
for r in wilcoxon_rows:
    print(f"  {r['comparison']:<52}: W={r['W']:.0f}, p={r['p_wilcoxon']:.4f}, "
          f"LayerCAM wins {r['n_layercam_wins']}/{r['n_pairs_total']} pairs")
print('Figure 9 saved.')

## 22. Figure 10 — Combined Publication Figure

In [ ]:
fig = plt.figure(figsize=(24, 17))
gs  = fig.add_gridspec(3, 4, hspace=0.52, wspace=0.40)
fig.suptitle(
    '2×2 Factorial CAM Analysis on PanNuke — DenseNet169\n'
    'Resolution Strategy (single / pyramid) × Gradient Treatment (GradCAM / LayerCAM)',
    fontsize=14, fontweight='bold', y=1.01,
)

# ── (A) IoU@0.50 heatmap ──────────────────────────────────────────────────────
ax_a = fig.add_subplot(gs[0, :2])
df50_h = df_summary[df_summary.threshold == 0.50].set_index('method')
mat_a  = np.array([
    [df50_h.loc[mlabel(m), f'iou_{ck}'] if mlabel(m) in df50_h.index else np.nan
     for ck in CLASS_KEYS]
    for m in METHODS
], dtype=float)
sns.heatmap(
    mat_a, ax=ax_a,
    xticklabels=CLASS_DISPLAY,
    yticklabels=[mlabel(m) for m in METHODS],
    cmap='YlOrRd', annot=True, fmt='.3f', annot_kws={'size': 9},
    linewidths=0.4, linecolor='#cccccc',
    cbar_kws={'label': 'IoU@0.50', 'shrink': 0.8},
)
ax_a.set_title('(A) Per-class IoU@0.50 — 50 test images', fontsize=10, fontweight='bold')
ax_a.tick_params(axis='x', rotation=20, labelsize=8)
ax_a.tick_params(axis='y', rotation=0,  labelsize=8)

# ── (B) IoU vs cardinality ────────────────────────────────────────────────────
ax_b = fig.add_subplot(gs[0, 2:])
for mkey in METHODS:
    cd = pooled_iou_by_cardinality(df_full, mkey)
    ax_b.plot(
        cd['cardinality'], cd['mean'],
        f"{mmarker(mkey)}-",
        color=mcolor(mkey), linewidth=2.2, markersize=8,
        label=mlabel(mkey),
    )
ax_b.set_xlabel('Label Cardinality', fontsize=9)
ax_b.set_ylabel('Mean IoU@0.50', fontsize=9)
ax_b.set_title('(B) IoU vs Co-occurrence Complexity', fontsize=10, fontweight='bold')
ax_b.legend(fontsize=8, framealpha=0.7)
ax_b.grid(alpha=0.2, linestyle=':')

# ── (C–F) 2×2 interference matrices ──────────────────────────────────────────
for idx, (ri, ci, mkey) in enumerate([(1, 0, 'std_gradcam'), (1, 1, 'fpn_gradcam'),
                                        (1, 2, 'std_layercam'), (1, 3, 'fpn_layercam')]):
    ax  = fig.add_subplot(gs[ri, ci])
    mat = int_matrices[mkey]
    sns.heatmap(
        mat, ax=ax,
        mask=np.eye(NUM_CLASSES, dtype=bool),
        cmap='RdBu_r', center=0, vmin=-vmax, vmax=vmax,
        annot=True, fmt='.2f', annot_kws={'size': 7.5},
        xticklabels=[s[:8] for s in CLASS_DISPLAY],
        yticklabels=[s[:8] for s in CLASS_DISPLAY],
        linewidths=0.3, linecolor='#cccccc', cbar=False,
    )
    letter = chr(67 + idx)   # C, D, E, F
    ax.set_title(f'({letter}) {mlabel(mkey)}', fontsize=9, fontweight='bold')
    ax.tick_params(axis='x', rotation=30, labelsize=7)
    ax.tick_params(axis='y', rotation=0,  labelsize=7)

# ── (G) Robustness gain — single layer ────────────────────────────────────────
ax_g = fig.add_subplot(gs[2, :2])
sub_g = pivot.dropna(subset=['gain_single']).sort_values('gain_single')
if not sub_g.empty:
    ax_g.barh(
        range(len(sub_g)), sub_g['gain_single'].values,
        color=['#388E3C' if v > 0 else '#C62828' for v in sub_g['gain_single'].values],
        alpha=0.85, edgecolor='white', linewidth=0.4,
    )
    ax_g.set_yticks(range(len(sub_g)))
    ax_g.set_yticklabels(
        [f"{r['class_c'][:9]}→{r['class_cp'][:9]}" for _, r in sub_g.iterrows()],
        fontsize=6.5,
    )
ax_g.axvline(0, color='black', linewidth=1)
ax_g.set_xlabel('Std GradCAM − Std LayerCAM  (positive = LayerCAM wins)', fontsize=8)
ax_g.set_title('(G) Single-Layer: Robustness Gain', fontsize=10, fontweight='bold')
ax_g.grid(axis='x', alpha=0.2, linestyle=':')

# ── (H) Robustness gain — FPN ─────────────────────────────────────────────────
ax_h = fig.add_subplot(gs[2, 2:])
sub_h = pivot.dropna(subset=['gain_fpn']).sort_values('gain_fpn')
if not sub_h.empty:
    ax_h.barh(
        range(len(sub_h)), sub_h['gain_fpn'].values,
        color=['#388E3C' if v > 0 else '#C62828' for v in sub_h['gain_fpn'].values],
        alpha=0.85, edgecolor='white', linewidth=0.4,
    )
    ax_h.set_yticks(range(len(sub_h)))
    ax_h.set_yticklabels(
        [f"{r['class_c'][:9]}→{r['class_cp'][:9]}" for _, r in sub_h.iterrows()],
        fontsize=6.5,
    )
ax_h.axvline(0, color='black', linewidth=1)
ax_h.set_xlabel('FPN-GradCAM − FPN-LayerCAM  (positive = FPN-LayerCAM wins)', fontsize=8)
ax_h.set_title('(H) Feature Pyramid: Robustness Gain', fontsize=10, fontweight='bold')
ax_h.grid(axis='x', alpha=0.2, linestyle=':')

plt.savefig(FIG_DIR / 'fig10_combined_main.pdf', bbox_inches='tight', dpi=300)
plt.savefig(FIG_DIR / 'fig10_combined_main.png', bbox_inches='tight', dpi=300)
plt.show()
print('Figure 10 (combined main) saved.')

## 23. Final Statistical Summary

In [ ]:
sep = '=' * 70
print(sep)
print('FINAL STATISTICAL SUMMARY — 2×2 FACTORIAL CAM COMPARISON')
print(sep)

print('\n(1) Macro IoU@0.50 on 50 test images:')
for mkey in METHODS:
    macro = float(np.nanmean([
        df_test[f'{mkey}_iou_0.50_{ck}'].dropna().mean() for ck in CLASS_KEYS
    ]))
    print(f'  {mlabel(mkey):<30}: {macro:.4f}')

print('\n(2) Factorial axis effects at IoU@0.50 (macro):')
sg  = float(np.nanmean([df_test[f'std_gradcam_iou_0.50_{ck}'].dropna().mean()  for ck in CLASS_KEYS]))
sl  = float(np.nanmean([df_test[f'std_layercam_iou_0.50_{ck}'].dropna().mean() for ck in CLASS_KEYS]))
fg  = float(np.nanmean([df_test[f'fpn_gradcam_iou_0.50_{ck}'].dropna().mean()  for ck in CLASS_KEYS]))
fl  = float(np.nanmean([df_test[f'fpn_layercam_iou_0.50_{ck}'].dropna().mean() for ck in CLASS_KEYS]))
effect_layer = ((sl - sg) + (fl - fg)) / 2
effect_pyr   = ((fg - sg) + (fl - sl)) / 2
interaction  = (fl - fg) - (sl - sg)
print(f'  LayerCAM effect (avg over resolution levels): {effect_layer:+.4f}')
print(f'  Pyramid effect  (avg over gradient methods) : {effect_pyr:+.4f}')
print(f'  Interaction term (Pyr×Layer - Pyr×Grad)     : {interaction:+.4f}')

print('\n(3) Spearman monotonicity (cardinality vs IoU@0.50, full dataset):')
for mkey in METHODS:
    cd      = pooled_iou_by_cardinality(df_full, mkey)
    rho, p  = stats.spearmanr(cd['cardinality'], cd['mean'])
    print(f'  {mlabel(mkey):<30}: rho={rho:+.4f}, p={p:.4f}')

print('\n(4) Significant interference pairs (permutation test p<0.05):')
for mkey in METHODS:
    sub  = df_perm[df_perm.method == mkey]
    sig  = sub['significant'].sum()
    n_pos = ((sub['significant']) & (sub['observed_delta'] > 0)).sum()
    n_neg = ((sub['significant']) & (sub['observed_delta'] < 0)).sum()
    print(f'  {mlabel(mkey):<30}: {sig}/{len(sub)}  '
          f'({n_pos} positive, {n_neg} negative)')

print('\n(5) Wilcoxon: LayerCAM reduces interference vs GradCAM?')
for la, lb in [('std_gradcam', 'std_layercam'), ('fpn_gradcam', 'fpn_layercam')]:
    valid = pivot.dropna(subset=[la, lb])
    if len(valid) >= 5:
        try:
            W, p = stats.wilcoxon(valid[la].values, valid[lb].values, alternative='greater')
            wins = int((valid[la].values > valid[lb].values).sum())
            print(f'  {mlabel(la):<30} > {mlabel(lb):<30}: '
                  f'W={W:.0f}, p={p:.4f}  ({wins}/{len(valid)} pairs)')
        except ValueError:
            print(f'  {mlabel(la)} vs {mlabel(lb)}: insufficient variation for test')

print('\n(6) Negative interference pairs (significant, delta < 0):')
neg = df_pairwise[
    df_pairwise['significant'] & (df_pairwise['delta_iou'] < 0)
][['method','class_c','class_cp','delta_iou','p_val','n_absent','n_present']]
if neg.empty:
    print('  None found.')
else:
    print(neg.sort_values('delta_iou').to_string(index=False))

print(f'\n{sep}')

## 24. Output File Manifest

In [ ]:
print('Output files in', OUT_DIR)
print('=' * 65)
all_files = sorted(OUT_DIR.rglob('*'))
for f in all_files:
    if f.is_file():
        rel  = str(f.relative_to(OUT_DIR))
        size = f.stat().st_size
        unit = 'KB' if size < 1_000_000 else 'MB'
        val  = size / 1_000 if size < 1_000_000 else size / 1_000_000
        print(f'  {rel:<58} {val:>7.1f} {unit}')

n_trip = sum(1 for _ in TRIP_DIR.rglob('*.png'))
print(f'\nTriplet figures : {n_trip}  ({N_TEST} images × {len(METHODS)} methods)')
print(f'Publication figs: 10  (fig1–fig10, PDF + PNG @ 300 DPI in {FIG_DIR.name}/)')